<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #6a11cb 0%, #2575fc 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎬 LTX-2.3 22B Distilled 1.1 Q4 — Video Generator</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Kaggle T4 GPU Edition — Created by <strong>AIQUEST Academy</strong></h3>
  <p style='color: #ddd; margin: 0; text-align: center;'>Text-to-Video • Image-to-Video • First & Last Frame | Wan2GP Engine + mmgp Profile 4</p>
</div>

---

<div align="center">
  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Kaggle-T4%20GPU-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white" />
  <br>
  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
</div>

---

### What is this notebook?

**Text-to-Video & Image-to-Video Generation** using **LTX-2.3 22B Distilled 1.1 Q4_K_M**

**GPU:** T4 x1 | **Model:** LTX-2.3 22B Distilled **1.1** Q4_K_M (~17.8GB, streamed via mmgp)
**Modes:** Text-to-Video • Image-to-Video • First Frame • Last Frame • First+Last Frame
**Pipeline:** Two-stage distilled 1.1 (8 steps half-res → 2× upscale → 3 steps full-res)
| VRAM | 16GB T4 VRAM |
| RAM | ~30GB Kaggle RAM |
| Modes | Text-to-Video, Image-to-Video (start/end frame) |

### Quick Start
1. **Settings → Accelerator → GPU T4 x1**
2. **Turn on Internet** in Settings sidebar
3. Run all cells in order
4. Upload a portrait image + audio file in the Gradio UI


---
## Step 1: Environment Setup

Optimizes memory for Kaggle T4 GPU (~30GB RAM, 16GB VRAM).

In [1]:
import os, gc, psutil

print('=== Kaggle T4 Environment Setup ===')
print(f'RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB total, {psutil.virtual_memory().available / 1024**3:.1f} GB available')

os.system('echo 3 | sudo tee /proc/sys/vm/drop_caches > /dev/null 2>&1')
os.system('echo 1 | sudo tee /proc/sys/vm/overcommit_memory > /dev/null 2>&1')
gc.collect()

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,garbage_collection_threshold:0.6'
os.environ['MALLOC_TRIM_THRESHOLD_'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print('✅ Environment optimized!')
print('   Kaggle has ~30GB RAM — no swap needed.')

=== Kaggle T4 Environment Setup ===
RAM: 31.4 GB total, 30.2 GB available
✅ Environment optimized!
   Kaggle has ~30GB RAM — no swap needed.


---
## Step 2: Clone Wan2GP & Install Dependencies

In [2]:
import subprocess
try:
    subprocess.run(['nvidia-smi'], check=True)
    print('GPU Active!')
except Exception:
    print('WARNING: No GPU. Go to Settings → Accelerator → GPU T4 x1')

!git clone https://github.com/DeepBeepMeep/Wan2GP.git

# ── PyTorch: Kaggle ships 2.4+ which dropped sm_60 (P100) CUDA kernels. ──
# ── PyTorch 2.3.1 is the last release with full sm_60 support.           ──
# ── Install it FIRST, then all other deps.                                ──
!pip install -q torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 \
    --index-url https://download.pytorch.org/whl/cu121
# ── After the cell above runs, click Kernel → Restart & Run All.          ──

!pip install --timeout 120 --retries 5 -q -r Wan2GP/requirements.txt
!pip install --timeout 120 --retries 5 -q mmgp gradio gguf soundfile

Sun Jun 14 12:20:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

---
## Step 3: Download All Required Models

Downloads GGUF Q4_K_M transformer + companion files. Large files → `/kaggle/tmp` (symlinked).

In [3]:
import os
from huggingface_hub import hf_hub_download

REPO = 'Abiray/LTX-2.3-22B-DISTILLED-1.1-GGUF'
COMPANION_REPO = 'DeepBeepMeep/LTX-2'
MODEL_DIR = 'Wan2GP/models'
TMP_DIR = '/kaggle/tmp/models'
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

# Distilled 1.1 Q4_K_M transformer from Abiray
TRANSFORMER_FILE    = 'LTX-2.3-22B-distilled-1.1-Q4_K_M.gguf'
TRANSFORMER_RENAMED = 'ltx-2.3-22b-distilled-1.1-Q4_K_M.gguf'

# Download transformer from Abiray
dest = os.path.join(MODEL_DIR, TRANSFORMER_RENAMED)
if os.path.exists(dest):
    print(f'  \u2713 Already exists: {TRANSFORMER_RENAMED}')
else:
    print(f'Downloading {TRANSFORMER_FILE} from {REPO} \u2192 /kaggle/tmp ...')
    hf_hub_download(repo_id=REPO, filename=TRANSFORMER_FILE, local_dir=TMP_DIR)
    actual = os.path.join(TMP_DIR, TRANSFORMER_FILE)
    os.symlink(actual, dest)
    print(f'  \u2713 {TRANSFORMER_RENAMED} (symlinked)')

# Companion large files from DeepBeepMeep
LARGE_FILES_COMPANION = [
    'ltx-2.3-22b-distilled-lora-384.safetensors',
    'ltx-2.3-22b_embeddings_connector.safetensors',
    'ltx-2.3-22b_text_embedding_projection.safetensors',
    'ltx-2.3-22b_vae.safetensors',
]

for f in LARGE_FILES_COMPANION:
    dest = os.path.join(MODEL_DIR, f)
    if os.path.exists(dest):
        print(f'  ✓ Already exists: {f}')
        continue
    print(f'Downloading {f} → /kaggle/tmp ...')
    hf_hub_download(repo_id=COMPANION_REPO, filename=f, local_dir=TMP_DIR)
    actual = os.path.join(TMP_DIR, f)
    os.symlink(actual, dest)
    print(f'  ✓ {f} (symlinked)')

# Small files download directly
SMALL_FILES = [
    'ltx-2.3-22b_audio_vae.safetensors',                 # 107 MB
    'ltx-2.3-22b_vocoder.safetensors',                   # 258 MB
    'ltx-2.3-spatial-upscaler-x2-1.1.safetensors',       # 996 MB
    'ltx-2.3-temporal-upscaler-x2-1.0.safetensors',      # 262 MB
]

for f in SMALL_FILES:
    dest = os.path.join(MODEL_DIR, f)
    if os.path.exists(dest):
        print(f'  ✓ Already exists: {f}')
        continue
    print(f'Downloading {f}...')
    hf_hub_download(repo_id=COMPANION_REPO, filename=f, local_dir=MODEL_DIR)
    print(f'  ✓ {f}')

# Gemma text encoder → /kaggle/tmp
GEMMA_FOLDER = 'gemma-3-12b-it-qat-q4_0-unquantized'
GEMMA_FILES = [
    'gemma-3-12b-it-qat-q4_0-unquantized_quanto_bf16_int8.safetensors',
    'added_tokens.json', 'chat_template.json', 'config_light.json',
    'generation_config.json', 'preprocessor_config.json', 'processor_config.json',
    'special_tokens_map.json', 'tokenizer.json', 'tokenizer.model', 'tokenizer_config.json',
]

gemma_dest = os.path.join(MODEL_DIR, GEMMA_FOLDER)
gemma_tmp = os.path.join(TMP_DIR, GEMMA_FOLDER)
if os.path.exists(gemma_dest):
    print(f'  ✓ Already exists: {GEMMA_FOLDER}/')
else:
    os.makedirs(gemma_tmp, exist_ok=True)
    for gf in GEMMA_FILES:
        tmp_file = os.path.join(gemma_tmp, gf)
        if os.path.exists(tmp_file): continue
        print(f'Downloading gemma/{gf} → /kaggle/tmp ...')
        hf_hub_download(repo_id=COMPANION_REPO, filename=f'{GEMMA_FOLDER}/{gf}', local_dir=TMP_DIR)
    os.symlink(gemma_tmp, gemma_dest)
    print(f'  ✓ {GEMMA_FOLDER}/ (symlinked)')

# Cleanup HF cache
import shutil
for d in [os.path.join(MODEL_DIR, '.cache'), os.path.join(TMP_DIR, '.cache')]:
    if os.path.exists(d): shutil.rmtree(d)

os.system('df -h /kaggle/working /kaggle/tmp')
print('\n✅ All downloads complete!')

LTX-2.3-22B-distilled-1.1-Q4_K_M.gguf:   0%|          | 0.00/17.8G [00:00<?, ?B/s]

  ✓ ltx-2.3-22b-distilled-1.1-Q4_K_M.gguf (symlinked)


ltx-2.3-22b-distilled-lora-384.safetenso(…):   0%|          | 0.00/7.61G [00:00<?, ?B/s]

  ✓ ltx-2.3-22b-distilled-lora-384.safetensors (symlinked)


ltx-2.3-22b_embeddings_connector.safeten(…):   0%|          | 0.00/4.03G [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

gemma-3-12b-it-qat-q4_0-unquantized/toke(…):   0%|          | 0.00/33.4M [00:00<?, ?B/s]

gemma-3-12b-it-qat-q4_0-unquantized/toke(…):   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

  ✓ gemma-3-12b-it-qat-q4_0-unquantized/ (symlinked)
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  1.7G   18G   9% /kaggle/working
overlay         7.9T  6.9T  1.1T  88% /

✅ All downloads complete!


---
## Step 4: Write the Text/Image-to-Video Script

Creates `run_ltx_t2v.py` — the full pipeline: model loading, mmgp offloading, generation, and Gradio UI.

In [4]:
%%writefile run_ltx_t2v.py
import gc
import os
import sys
import json
import random
import tempfile
import glob
import traceback
import numpy as np
import subprocess
import psutil
import soundfile as sf
from PIL import Image

# ---- bootstrap Wan2GP ----
WAN2GP_DIR = os.path.abspath("Wan2GP")
sys.path.insert(0, WAN2GP_DIR)
os.chdir(WAN2GP_DIR)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128,garbage_collection_threshold:0.5"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"

import torch

# Detect GPU architecture once — used throughout to gate sm_60-specific workarounds
_GPU_SM = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0, 0)
_IS_SM60 = (_GPU_SM[0] == 6)   # P100 = sm_60; T4 = sm_75; A100 = sm_80; etc.
_IS_SM70_PLUS = (_GPU_SM[0] >= 7)

if _IS_SM60:
    # P100 (sm_60/Pascal) has NO native BF16 CUDA kernels and the current
    # PyTorch 2.4+ build dropped sm_60 support entirely.
    # Force FP16 to get as far as possible, and patch audio ops to run on CPU.
    os.environ["WGP_DTYPE"] = "fp16"
    print(f"  [GPU] sm_60 detected (P100) — FP16 mode + CPU audio patches enabled")
else:
    print(f"  [GPU] sm_{_GPU_SM[0]}{_GPU_SM[1]} detected — native CUDA mode, no patches needed")
import gradio as gr
from shared.utils.audio_video import save_video

# Lightweight Google Drive fallback for the generated standalone script.
# The full backup helpers are available in the notebook UI cells, but the
# standalone script must define these names on its own to avoid NameError.
GDRIVE = None
GDRIVE_FOLDER_ID = None
GDRIVE_AVAILABLE = False


def upload_video_to_gdrive(drive, video_path, folder_id=None):
    """Fallback uploader used by the generated standalone script."""
    return None


def backup_video_to_gdrive(selected_video):
    return "⚠️ Google Drive backup is disabled in this local run."


def backup_all_videos_to_gdrive():
    return "⚠️ Google Drive backup is disabled in this local run."


def delete_selected_with_gdrive_backup(selected):
    return list_outputs(), "⚠️ Google Drive backup is disabled in this local run."


def delete_all_with_gdrive_backup():
    return list_outputs(), "⚠️ Google Drive backup is disabled in this local run."


# ==== GGUF EXTENSION HANDLER ====
# mmgp uses an extension handler system for non-safetensors formats.
# The full Wan2GP app registers this internally; for standalone scripts
# we must register it ourselves before any model loading happens.

def _register_gguf_handler():
    """Register the GGUF handler with mmgp's quant_router."""
    import shared.qtypes.gguf
    print("  [GGUF] ✅ Extension handler registered with mmgp (Wan2GP native)")

def _patch_ltx2_config_loading():
    """Patch _load_config_from_checkpoint to handle GGUF metadata errors."""
    import models.ltx2.ltx2 as ltx2_mod
    _original = ltx2_mod._load_config_from_checkpoint

    def _patched(path, fallback_config_path=None):
        from mmgp import quant_router
        if isinstance(path, (list, tuple)):
            path = path[0] if path else ""
        if not path:
            return {}
        try:
            _, metadata = quant_router.load_metadata_state_dict(path)
            if metadata:
                config_raw = metadata.get("config")
                if config_raw:
                    config = ltx2_mod._normalize_config(config_raw)
                    if config:
                        return config
        except Exception as e:
            print(f"  [GGUF Patch] Metadata read: {type(e).__name__}, using fallback config")
        if fallback_config_path and os.path.isfile(fallback_config_path):
            try:
                with open(fallback_config_path, "r", encoding="utf-8") as f:
                    config = ltx2_mod._normalize_config(json.load(f))
                    if config:
                        print(f"  [GGUF Patch] ✅ Config loaded from {os.path.basename(fallback_config_path)}")
                        return config
            except Exception:
                pass
        return {}

    ltx2_mod._load_config_from_checkpoint = _patched
    print("  [GGUF Patch] ✅ Config loading patched for GGUF")


# ==== GPU INFO ====
print(f"GPU: {torch.cuda.get_device_name()}")
print(f"Compute Capability: {torch.cuda.get_device_capability()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
ram = psutil.virtual_memory()
print(f"RAM: {ram.total / 1024**3:.1f} GB total, {ram.available / 1024**3:.1f} GB available")
sys.stdout.flush()

torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

# ==== REGISTER GGUF + LOAD MODEL ====
print("\nLoading LTX-2.3 22B Distilled 1.1 (GGUF Q4_K_M)...")
sys.stdout.flush()

from mmgp import offload
from shared.utils import files_locator as fl

fl.set_checkpoints_paths(["models", "ckpts", "."])

from models.ltx2.ltx2_handler import family_handler

# Register GGUF handler + patch config loading BEFORE load_model
_register_gguf_handler()
_patch_ltx2_config_loading()


class _AudioEncoderP100Wrapper:
    """
    Wraps the audio encoder to fix cudaErrorNoKernelImageForDevice on P100.

    Root cause (ltx2.py line 1171-1176):
        audio_params = next(self.audio_encoder.parameters(), None)
        audio_device = audio_params.device   # returns 'cpu' under mmgp Profile 4
        mel = mel.to(device=audio_device, ...)  # mel goes to CPU
        audio_latent = self.audio_encoder(mel)  # mmgp moves encoder to CUDA
                                                # but mel is STILL on CPU
        -> Conv2d: weights on CUDA, input on CPU -> cudaErrorNoKernelImageForDevice

    Fix: intercept __call__ and move mel to CUDA FP16 BEFORE passing to encoder.
    All attribute access (sample_rate, mel_bins, parameters, etc.) is proxied.
    """
    def __init__(self, encoder):
        object.__setattr__(self, '_enc', encoder)

    def __call__(self, mel):
        if torch.cuda.is_available():
            mel = mel.to(
                device=torch.device("cuda", torch.cuda.current_device()),
                dtype=torch.float16,  # P100 (sm_60) has no BF16 CUDA kernels
            )
        return object.__getattribute__(self, '_enc')(mel)

    def __getattr__(self, name):
        return getattr(object.__getattribute__(self, '_enc'), name)

    def __setattr__(self, name, value):
        if name == '_enc':
            object.__setattr__(self, name, value)
        else:
            setattr(object.__getattribute__(self, '_enc'), name, value)

    def __repr__(self):
        return f"_AudioEncoderP100Wrapper({object.__getattribute__(self, '_enc')!r})"

base_model_type = "ltx2_22B"
model_def = {"ltx2_pipeline": "distilled"}
extra = family_handler.query_model_def(base_model_type, model_def)
model_def.update(extra)

gemma_folder = "models/gemma-3-12b-it-qat-q4_0-unquantized"
gemma_files = sorted(glob.glob(os.path.join(gemma_folder, "*.safetensors")))
quanto_files = [f for f in gemma_files if "quanto" in f]
text_encoder_file = quanto_files[0] if quanto_files else (gemma_files[0] if gemma_files else None)
if not text_encoder_file:
    raise FileNotFoundError(f"No .safetensors in {gemma_folder}. Check download cell.")
print(f"  Text encoder: {os.path.basename(text_encoder_file)}")

transformer_path = os.path.join("models", "ltx-2.3-22b-distilled-1.1-Q4_K_M.gguf")
if not os.path.isfile(transformer_path):
    raise FileNotFoundError(f"{transformer_path} missing. Check download cell.")
print(f"  Transformer : {os.path.basename(transformer_path)}")
sys.stdout.flush()

# P100 (Pascal sm_60) has NO native BF16 support.
# Model weights loaded in FP16; but runtime activations (mel spectrogram) that
# flow through the BF16 autocast of the transformer are patched via CausalConv2d above.
# VAE_dtype = float32 for the video VAE (safer, audio VAE is patched separately).
MODEL_DTYPE = torch.float16
VAE_DTYPE   = torch.float16  # FP16 for T4: 65 TFLOPS vs FP32 8 TFLOPS = ~8x faster VAE decode

ltx2_model, pipe = family_handler.load_model(
    model_filename=transformer_path,
    model_type="ltx2_22B_distilled",
    base_model_type=base_model_type,
    model_def=model_def,
    dtype=MODEL_DTYPE,
    VAE_dtype=VAE_DTYPE,
    text_encoder_filename=text_encoder_file,
)

# ==== Verify pipeline components ====
print("\n--- Pipeline Components ---")
for name, component in pipe.items():
    if component is not None:
        ctype = type(component).__name__
        if hasattr(component, 'parameters'):
            try:
                p = next(component.parameters())
                print(f"  {name}: {ctype} (dtype={p.dtype})")
            except StopIteration:
                print(f"  {name}: {ctype} (no params)")
        else:
            print(f"  {name}: {ctype}")
    else:
        print(f"  {name}: None")
sys.stdout.flush()

# ==== sm_60 (P100) Only: patch CausalConv2d BEFORE offload.profile() ====
# On sm_75+ (T4, A100, etc.) this is skipped — native CUDA handles everything.
# On sm_60 the entire PyTorch CUDA kernel set is unavailable (2.4+ dropped it),
# so F.pad + conv must run on CPU. This patch must be applied AFTER load_model
# (class importable) but BEFORE offload.profile (mmgp captures previous_method).
if _IS_SM60:
    try:
        import torch.nn.functional as _F
        from models.ltx2.ltx_core.model.audio_vae.causal_conv_2d import CausalConv2d as _CC2d

        def _cc2d_cpu_pad(self, x: torch.Tensor) -> torch.Tensor:
            if x.is_cuda:
                dev, dt = x.device, x.dtype
                x_cpu = x.detach().cpu().float()
                x_cpu = _F.pad(x_cpu, self.padding)
                w = self.conv.weight.detach().cpu().float()
                b = self.conv.bias.detach().cpu().float() if self.conv.bias is not None else None
                out = _F.conv2d(x_cpu, w, b,
                                self.conv.stride, self.conv.padding,
                                self.conv.dilation, self.conv.groups)
                return out.to(device=dev, dtype=dt)
            else:
                x = _F.pad(x, self.padding)
                return self.conv(x)

        _CC2d.forward = _cc2d_cpu_pad
        print("  [sm_60 Fix] ✅ CausalConv2d patched: pad+conv run on CPU")
    except Exception as _e:
        print(f"  [sm_60 Fix] ❌ Could not patch CausalConv2d: {_e}")
else:
    print("  [sm_60 Fix] ⏭️  Skipped (not sm_60)")

# ==== Apply mmgp Profile 4 ====
print("\nApplying mmgp Profile 4 with per-model budgets...")
sys.stdout.flush()

offload.profile(
    pipe,
    profile_no=4,
    quantizeTransformer=False,
    convertWeightsFloatTo=torch.float16,
    budgets={
        # Budget 6000: ~9 min total (sweet spot — steps ~10s, VAE decode ~4 min)
        "transformer":       6000,
        "text_encoder":      1500,
        "video_encoder":     2000,
        "video_decoder":     3000,
        "audio_encoder":     1000,
        "audio_decoder":     1000,
        "vocoder":           500,
        "spatial_upsampler": 1500,
        "vae":               1000,
        "*":                 1000,
    },
)
offload.shared_state["_attention"] = "sdpa"

print("\n✅ Setup complete! Distilled 1.1 Text/Image-to-Video pipeline active.")
sys.stdout.flush()

# ==== HELPER FUNCTIONS ====
def get_resolution(base_res_str, aspect_ratio_str):
    base_resolutions = {"1080p": 1088, "720p": 704, "540p": 544, "480p": 480}
    ratios = {
        "16:9 Landscape": 16/9, "4:3 Standard": 4/3,
        "1:1 Square": 1.0, "3:4 Portrait": 3/4, "9:16 Portrait": 9/16,
    }
    base = base_resolutions.get(base_res_str, 704)
    ratio = ratios.get(aspect_ratio_str, 16/9)
    if ratio >= 1.0:
        height = base
        width = int(base * ratio)
    else:
        width = base
        height = int(base / ratio)
    return (width // 32) * 32, (height // 32) * 32

def get_vae_tile_size(height, width):
    vram_mb = torch.cuda.get_device_properties(0).total_memory / (1024**2)
    effective_vram = vram_mb / 1.5
    if effective_vram >= 24000: vae_config = 1
    elif effective_vram >= 8000: vae_config = 2
    else: vae_config = 3
    if max(height, width) > 480: vae_config += 1
    if vae_config <= 1: return 0
    elif vae_config == 2: return 512
    elif vae_config == 3: return 256
    return 128

def snap_to_ltx_frames(duration_sec: float, fps: float = 24.0, max_frames: int = 721) -> int:
    """Convert audio duration (seconds) to nearest valid LTX frame count.
    LTX distilled requires frames = 8k+1 (1, 9, 17, 25, ... 721, ...).
    Caps at max_frames (default 721 = 30s @ 24fps).
    """
    raw = duration_sec * fps
    # round to nearest 8k+1
    k = max(0, round((raw - 1) / 8))
    frames = 8 * k + 1
    frames = max(49, min(frames, max_frames))   # floor at 2s (49f), cap at max
    return int(frames)

@torch.inference_mode()
def Video_Generation(prompt, input_image_start, input_image_end, seed, duration_dropdown,
                     resolution_dropdown, aspect_ratio_dropdown,
                     guide_scale=3.0, num_steps=8, progress=None):
    try:
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
        progress(0, desc="Starting...")

        duration_map = {
            "2 Seconds (49 frames)":  49,
            "3 Seconds (73 frames)":  73,
            "5 Seconds (121 frames)": 121,
            "8 Seconds (193 frames)": 193,
            "10 Seconds (241 frames)": 241,
            "15 Seconds (361 frames)": 361,
        }
        frame_rate = 24.0
        num_frames = duration_map.get(duration_dropdown, 121)
        width, height = get_resolution(resolution_dropdown, aspect_ratio_dropdown)

        if seed is None or seed < 0:
            seed = random.randint(0, 2**32 - 1)
        seed = int(seed)

        image_start = None
        image_end   = None
        if input_image_start is not None:
            image_start = Image.open(input_image_start).convert("RGB")
        if input_image_end is not None:
            image_end = Image.open(input_image_end).convert("RGB")

        free_vram = torch.cuda.mem_get_info()[0] / 1024**3
        ram = psutil.virtual_memory()
        mode = "T2V" if image_start is None else ("I2V first+last" if image_end else "I2V start")
        print(f"\n{'='*60}")
        print(f"Generating [{mode}]: {width}x{height}, {num_frames} frames, seed={seed}")
        print(f"  VRAM free: {free_vram:.2f} GB | RAM free: {ram.available / 1024**3:.1f} GB")
        print(f"  Prompt: {prompt[:120]}{'...' if len(prompt) > 120 else ''}")
        print(f"  Guide scale: {guide_scale}")
        print(f"{'='*60}")
        sys.stdout.flush()

        # Hardcode VAE tile size to 256 (matches audio notebook, prevents OOM/slowdowns at higher res)
        vae_tile_size = 256
        print(f"  VAE tile size: {vae_tile_size} (fixed)")

        total_steps = [8]
        current_step = [0]
        current_pass = [1]

        def cb(step, latent, is_start, override_num_inference_steps=None, pass_no=None, **kwargs):
            if is_start:
                if override_num_inference_steps is not None:
                    total_steps[0] = override_num_inference_steps
                if pass_no is not None:
                    current_pass[0] = pass_no
                current_step[0] = 0
                return
            current_step[0] += 1
            stage_name = (
                "Stage 1 (half-res)" if current_pass[0] == 1
                else "Stage 2 (full-res refine)" if current_pass[0] == 2
                else "Diffusion"
            )
            free_v = torch.cuda.mem_get_info()[0] / 1024**3
            print(f"  [{stage_name}] step {current_step[0]}/{total_steps[0]} | VRAM free: {free_v:.2f} GB")
            sys.stdout.flush()
            frac = current_step[0] / max(total_steps[0], 1)
            if current_pass[0] == 2:
                frac = 0.73 + 0.22 * frac
            else:
                frac = frac * 0.73
            progress(min(frac, 0.95), desc=f"{stage_name}: {current_step[0]}/{total_steps[0]}")

        _stage_labels = {
            "VAE Encoding": ("🇯️  VAE Encoding input frames...",   0.05),
            "VAE Decoding": ("🎦 VAE Decoding latents → frames...", 0.88),
            "Upsampling":   ("🔭 Spatial upsampling latents...",  0.83),
        }
        import time as _time
        _t = [_time.time()]

        def set_progress_status(status: str):
            dt = _time.time() - _t[0]; _t[0] = _time.time()
            label, frac = _stage_labels.get(status, (f"⏳ {status}...", 0.85))
            print(f"  [{status}] {label}  (+{dt:.1f}s)")
            sys.stdout.flush()
            progress(frac, desc=label)

        gen_kwargs = dict(
            input_prompt=prompt,
            image_start=image_start,
            height=height,
            width=width,
            frame_num=num_frames,
            fps=frame_rate,
            seed=seed,
            callback=cb,
            VAE_tile_size=vae_tile_size,
            input_video_strength=1.0,
            denoising_strength=1.0,
            guide_scale=float(guide_scale),
            sampling_steps=int(num_steps),
            guide_phases=2,
            n_prompt="",
            enhance_prompt=False,
            video_prompt_type="",
            audio_prompt_type="",
            set_progress_status=set_progress_status,
        )
        if image_end is not None:
            gen_kwargs["image_end"] = image_end

        print("  Diffusion starting → Stage 1 → spatial upsample → Stage 2 → VAE decode...")
        sys.stdout.flush()
        _t[0] = _time.time()

        result = ltx2_model.generate(**gen_kwargs)

        progress(0.97, desc="✅ Generation done — saving video...")
        print("  Pipeline complete.")
        sys.stdout.flush()

        if isinstance(result, dict):
            video_tensor = result.get("x")
            audio_data   = result.get("audio")
            audio_sr     = result.get("audio_sampling_rate", 24000)
        elif isinstance(result, tuple):
            video_tensor = result[0]
            audio_data   = result[1] if len(result) > 1 else None
            audio_sr     = result[2] if len(result) > 2 else 24000
        else:
            video_tensor = result
            audio_data, audio_sr = None, 24000

        if video_tensor is None or not torch.is_tensor(video_tensor):
            return None, f"❌ No video tensor. Got: {type(video_tensor)}"

        print(f"  Video tensor: {video_tensor.shape}, dtype={video_tensor.dtype}")
        video_tensor = video_tensor.cpu()
        gc.collect(); torch.cuda.empty_cache()

        OUTPUT_DIR = "/kaggle/working/outputs"
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        timestamp = str(int(__import__("time").time()))
        out_path = os.path.join(OUTPUT_DIR, f"ltx_video_{timestamp}.mp4")
        video_for_save = video_tensor.unsqueeze(0).float() / 127.5 - 1.0
        save_video(tensor=video_for_save, save_file=out_path, fps=frame_rate, normalize=True, value_range=(-1, 1))
        print(f"  ✅ Video saved: {out_path}")

        # ==== Mux native model audio (if generated) ====
        if audio_data is not None:
            try:
                import soundfile as sf
                audio_tmp = tempfile.mktemp(suffix=".wav")
                if isinstance(audio_data, np.ndarray):
                    audio_np = audio_data
                    if audio_np.ndim == 2 and audio_np.shape[0] <= 2:
                        audio_np = audio_np.T
                    sf.write(audio_tmp, audio_np, int(audio_sr or 24000))
                elif torch.is_tensor(audio_data):
                    import torchaudio
                    cpu_audio = audio_data.cpu().float()
                    if cpu_audio.dim() == 1: cpu_audio = cpu_audio.unsqueeze(0)
                    if cpu_audio.dim() == 3: cpu_audio = cpu_audio.squeeze(0)
                    torchaudio.save(audio_tmp, cpu_audio, int(audio_sr or 24000))
                else:
                    raise ValueError(f"Unknown audio type: {type(audio_data)}")

                final_path = out_path.replace(".mp4", "_with_audio.mp4")
                subprocess.run([
                    "ffmpeg", "-y", "-i", out_path, "-i", audio_tmp,
                    "-c:v", "copy", "-c:a", "aac", "-b:a", "192k",
                    "-shortest", final_path
                ], check=True, capture_output=True)
                if os.path.exists(final_path) and os.path.getsize(final_path) > 0:
                    out_path = final_path
                    print(f"  ✅ Native audio muxed into output")
                else:
                    print(f"  ⚠️ Audio mux produced empty file, using video-only")
            except Exception as e:
                print(f"  ⚠️ Audio mux failed: {e}")

        del video_tensor, video_for_save
        gc.collect(); torch.cuda.empty_cache()
        progress(1.0, desc="Done!")
        return out_path, f"✅ Done! Seed: {seed} | {width}x{height} | {num_frames} frames"

    except Exception as e:
        traceback.print_exc()
        gc.collect(); torch.cuda.empty_cache()
        return None, f"❌ Error: {str(e)}"

# ==== GRADIO UI (AIQUEST BRANDED) ====
CSS = """@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1000px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #6a11cb 0%, #2575fc 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(102,126,234,0.3); }
.brand-title { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 16px; }
.social-buttons { display: flex; justify-content: center; gap: 12px; flex-wrap: wrap; }
.social-btn { padding: 10px 24px; border-radius: 8px; font-weight: 700; font-size: 15px; text-decoration: none; display: inline-block; color: white; transition: all 0.3s; box-shadow: 0 4px 12px rgba(0,0,0,0.2); }
.social-btn:hover { transform: translateY(-2px); box-shadow: 0 6px 16px rgba(0,0,0,0.3); }
.youtube-btn { background: linear-gradient(135deg, #FF0000 0%, #CC0000 100%); }
.x-btn { background: linear-gradient(135deg, #000000 0%, #333333 100%); }
button.primary { background: linear-gradient(135deg, #6a11cb 0%, #2575fc 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#stop-btn { background: linear-gradient(135deg, #ef4444 0%, #b91c1c 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#clear-btn { background: linear-gradient(135deg, #6b7280 0%, #374151 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
.footer { text-align: center; padding: 20px; margin-top: 30px; border-top: 2px solid #e5e7eb; color: #6b7280; }
"""


# ═══════════════════════════════════════════════════════════════════════════════
#  BULK PROCESSING HELPERS
# ═══════════════════════════════════════════════════════════════════════════════
import csv, io, zipfile, shutil, time as _time, threading

_bulk_stop_flag = threading.Event()

def _parse_bulk_csv(csv_text: str):
    """Parse CSV with columns: prompt, start_image, end_image (optional),
    duration, resolution, aspect_ratio, seed, guide_scale, steps.
    Returns list of dicts."""
    jobs = []
    reader = csv.DictReader(io.StringIO(csv_text.strip()))
    # normalise header names (strip spaces, lower)
    for row in reader:
        row = {k.strip().lower(): v.strip() for k, v in row.items()}
        jobs.append({
            "prompt":       row.get("prompt", ""),
            "start_image":  row.get("start_image", row.get("start image", "")),
            "end_image":    row.get("end_image",   row.get("end image",   "")),
            "duration":     row.get("duration",    "5 Seconds (121 frames)"),
            "resolution":   row.get("resolution",  "720p"),
            "aspect_ratio": row.get("aspect_ratio", row.get("aspect ratio", "16:9 Landscape")),
            "seed":         int(row.get("seed", -1)),
            "guide_scale":  float(row.get("guide_scale", row.get("guide scale", 3.0))),
            "steps":        int(row.get("steps", 8)),
        })
    return jobs

def _extract_images_zip(zip_path: str, dest_dir: str):
    """Unzip image archive into dest_dir, return {filename: full_path} map."""
    os.makedirs(dest_dir, exist_ok=True)
    img_map = {}
    if zip_path and os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path) as zf:
            for name in zf.namelist():
                if name.lower().endswith((".png", ".jpg", ".jpeg", ".webp")):
                    zf.extract(name, dest_dir)
                    base = os.path.basename(name)
                    img_map[base] = os.path.join(dest_dir, name)
    return img_map

def run_bulk_queue(csv_file, zip_file,
                   default_duration, default_resolution,
                   default_aspect, default_guide, default_steps,
                   progress=None):
    """Main bulk runner called by the Gradio button."""
    global _bulk_stop_flag
    _bulk_stop_flag.clear()

    if csv_file is None:
        return None, "❌ Please upload a CSV file."

    # ── Read CSV ──────────────────────────────────────────────────────────────
    try:
        with open(csv_file, "r", encoding="utf-8-sig") as f:
            csv_text = f.read()
        jobs = _parse_bulk_csv(csv_text)
    except Exception as e:
        return None, f"❌ CSV parse error: {e}"

    if not jobs:
        return None, "❌ No jobs found in CSV."

    # ── Extract image zip ─────────────────────────────────────────────────────
    img_dir = "/kaggle/working/bulk_images"
    img_map = _extract_images_zip(zip_file, img_dir) if zip_file else {}

    total  = len(jobs)
    passed = []
    failed = []
    log_lines = [f"🚀 Starting {total} job(s)…\n"]

    for idx, job in enumerate(jobs):
        if _bulk_stop_flag.is_set():
            log_lines.append("🛑 Stopped by user.")
            break

        pct = idx / total
        progress(pct, desc=f"Job {idx+1}/{total}: {job['prompt'][:50]}…")
        log_lines.append(f"\n[{idx+1}/{total}] {job['prompt'][:80]}")

        # resolve image paths
        def _resolve(name):
            if not name:
                return None
            # absolute or relative to /kaggle/working
            for candidate in [name,
                               os.path.join("/kaggle/working", name),
                               img_map.get(os.path.basename(name), "")]:
                if candidate and os.path.exists(candidate):
                    return candidate
            return None

        start_path = _resolve(job["start_image"])
        end_path   = _resolve(job["end_image"])

        # apply per-job overrides or fall back to UI defaults
        duration  = job["duration"]    or default_duration
        res       = job["resolution"]  or default_resolution
        aspect    = job["aspect_ratio"]or default_aspect
        gscale    = job["guide_scale"] or default_guide
        steps     = job["steps"]       or default_steps
        seed      = job["seed"]

        try:
            out_path, status = Video_Generation(
                prompt              = job["prompt"],
                input_image_start   = start_path,
                input_image_end     = end_path,
                seed                = seed,
                duration_dropdown   = duration,
                resolution_dropdown = res,
                aspect_ratio_dropdown = aspect,
                guide_scale         = gscale,
                num_steps           = steps,
            )
            passed.append(out_path)
            log_lines.append(f"  ✅ {status}")
        except Exception as e:
            failed.append(str(e))
            log_lines.append(f"  ❌ Error: {e}")

        gc.collect(); torch.cuda.empty_cache()

    progress(1.0, desc="Done!")
    summary = (f"\n\n{'═'*50}\n"
               f"✅ Completed: {len(passed)}  ❌ Failed: {len(failed)}  "
               f"Total: {total}\n{'═'*50}")
    log_lines.append(summary)

    # Clean up extracted images to save space
    if os.path.exists(img_dir):
        shutil.rmtree(img_dir, ignore_errors=True)

    last_video = passed[-1] if passed else None
    return last_video, "\n".join(log_lines)

def stop_bulk():
    _bulk_stop_flag.set()
    return "🛑 Stop signal sent — current job will finish, then queue halts."

OUTPUT_DIR = "/kaggle/working/outputs"

def list_outputs():
    if not os.path.isdir(OUTPUT_DIR):
        return []
    videos = [f for f in os.listdir(OUTPUT_DIR) if f.lower().endswith(('.mp4', '.mkv', '.webm'))]
    videos.sort(key=lambda x: os.path.getmtime(os.path.join(OUTPUT_DIR, x)), reverse=True)
    return videos


def delete_selected_output(selected):
    if not selected:
        return list_outputs(), "❌ No video selected."
    path = os.path.join(OUTPUT_DIR, selected)
    if os.path.exists(path):
        try:
            os.remove(path)
            return list_outputs(), f"✅ Deleted {selected}"
        except Exception as e:
            return list_outputs(), f"❌ Delete failed: {e}"
    return list_outputs(), "❌ File not found."


def delete_all_outputs():
    if not os.path.isdir(OUTPUT_DIR):
        return [], "✅ No outputs to delete."
    for f in os.listdir(OUTPUT_DIR):
        if f.lower().endswith(('.mp4', '.mkv', '.webm')):
            try:
                os.remove(os.path.join(OUTPUT_DIR, f))
            except Exception:
                pass
    return list_outputs(), "✅ All output videos deleted."

with gr.Blocks(css=CSS, theme=gr.themes.Soft(), title="LTX-2.3 22B Video Generator | AIQUEST") as demo:
    gr.HTML('<div class="brand-header"><div class="brand-title">🎬 LTX-2.3 22B Distilled 1.1 Q4 — Video Generator</div><div class="brand-subtitle">Created by <strong>AIQuest Academy</strong> | Kaggle T4 GPU</div><div class="social-buttons"><a href="https://youtube.com/@aiquestacademy" target="_blank" class="social-btn youtube-btn">▶️ Subscribe on YouTube</a><a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a></div></div>')

    with gr.Tabs():
        # ════════════════════════════════════════════════════════════════════════════════════
        # TAB 1: SINGLE VIDEO GENERATION
        # ════════════════════════════════════════════════════════════════════════════════════
        with gr.TabItem("🎬 Single Video", id="single"):
            gr.Markdown(
                "**Two-stage distilled 1.1 pipeline** (8 steps half-res → 2× upscale → 3 steps full-res) | Q4_K_M ~17.8GB\n\n"
                "💡 **Tip:** Leave image inputs empty for pure Text-to-Video. Add a start image for Image-to-Video."
            )

            with gr.Column():
                prompt = gr.Textbox(
                    label="🎨 Prompt", lines=3,
                    placeholder="A majestic eagle soaring over snowy mountain peaks at golden hour, cinematic, 4K..."
                )

                with gr.Accordion("🖼️ Image to Video (Optional)", open=False):
                    with gr.Row():
                        input_image_start = gr.Image(type="filepath", label="🎬 Start Frame (First Frame)")
                        input_image_end   = gr.Image(type="filepath", label="🎬 End Frame (Last Frame — Optional)")
                    gr.Markdown(
                        "*• **Start Frame only** → Image-to-Video (model generates from your image)*\n"
                        "*• **Both frames** → First+Last frame interpolation (model generates in-between)*\n"
                        "*• **Neither** → Pure Text-to-Video*"
                    )

                with gr.Row():
                    seed = gr.Number(label="🎲 Seed (-1 for Random)", value=-1, precision=0)
                    duration_dropdown = gr.Dropdown(
                        label="⏱️ Duration",
                        choices=[
                            "2 Seconds (49 frames)",
                            "3 Seconds (73 frames)",
                            "5 Seconds (121 frames)",
                            "8 Seconds (193 frames)",
                            "10 Seconds (241 frames)",
                            "15 Seconds (361 frames)",
                        ],
                        value="5 Seconds (121 frames)",
                    )

                with gr.Row():
                    resolution_dropdown = gr.Dropdown(
                        label="📐 Base Resolution",
                        choices=["1080p", "720p", "540p", "480p"],
                        value="720p",
                    )
                    aspect_ratio_dropdown = gr.Dropdown(
                        label="📏 Aspect Ratio",
                        choices=["16:9 Landscape", "4:3 Standard", "1:1 Square", "3:4 Portrait", "9:16 Portrait"],
                        value="16:9 Landscape",
                    )

                guide_scale = gr.Slider(
                    label="🎯 Prompt Strength (guide_scale)",
                    minimum=1.0, maximum=8.0, step=0.5, value=3.0,
                    info="3.0 = optimal for T2V | 4.0+ = strong prompt | 1–2 = free generation",
                )
                num_steps = gr.Slider(
                    label="⚡ Diffusion Steps",
                    minimum=2, maximum=8, step=1, value=8,
                    info="6 = faster | 8 = best quality (default)",
                )

                with gr.Row():
                    gen_btn   = gr.Button("🎬 Generate Video", variant="primary", size="lg", elem_id="gen-btn")
                    stop_btn  = gr.Button("🛑 Stop",            variant="secondary", size="lg", elem_id="stop-btn")
                    clear_btn = gr.Button("🗑️ Clear",           variant="secondary", size="lg", elem_id="clear-btn")
                video_out  = gr.Video(label="🎥 Output")
                status_out = gr.Textbox(label="ℹ️ Status", interactive=False)

                with gr.Accordion("🗂️ Output Manager", open=False):
                    gr.Markdown(f"**🔗 Google Drive Status:** {'✅ ENABLED - Videos will be backed up before deletion' if GDRIVE_AVAILABLE else '⚠️ DISABLED - Videos will be deleted without backup'}")
                    
                    refresh_outputs_btn = gr.Button("🔄 Refresh Outputs")
                    outputs_dropdown = gr.Dropdown(
                        label="Generated Videos",
                        choices=[],
                        interactive=True
                    )
                    
                    with gr.Row():
                        backup_btn = gr.Button("📤 Backup to Google Drive", variant="primary")
                        delete_output_btn = gr.Button("🗑️ Delete with Backup", variant="stop")
                    
                    with gr.Row():
                        backup_all_btn = gr.Button("📤 Backup All to Drive")
                        delete_all_btn = gr.Button("⚠️ Delete All (after backup)", variant="stop")
                    
                    delete_status = gr.Textbox(label="Status", interactive=False)

                    refresh_outputs_btn.click(
                        fn=list_outputs,
                        outputs=[outputs_dropdown]
                    )
                    
                    backup_btn.click(
                        fn=backup_video_to_gdrive,
                        inputs=[outputs_dropdown],
                        outputs=[delete_status]
                    )
                    
                    backup_all_btn.click(
                        fn=backup_all_videos_to_gdrive,
                        outputs=[delete_status]
                    )

                    delete_output_btn.click(
                        fn=delete_selected_with_gdrive_backup,
                        inputs=[outputs_dropdown],
                        outputs=[outputs_dropdown, delete_status]
                    )

                    delete_all_btn.click(
                        fn=delete_all_with_gdrive_backup,
                        outputs=[outputs_dropdown, delete_status]
                    )

                gen_event = gen_btn.click(
                    fn=Video_Generation,
                    inputs=[prompt, input_image_start, input_image_end, seed, duration_dropdown,
                            resolution_dropdown, aspect_ratio_dropdown, guide_scale, num_steps],
                    outputs=[video_out, status_out],
                )
                stop_btn.click(fn=None, cancels=[gen_event])
                clear_btn.click(
                    fn=lambda: (None, None, None, "", -1),
                    outputs=[input_image_start, input_image_end, video_out, prompt, seed],
                )

        # ════════════════════════════════════════════════════════════════════════════════════
        # TAB 2: BULK QUEUE GENERATION
        # ════════════════════════════════════════════════════════════════════════════════════
        with gr.TabItem("📦 Bulk Queue", id="bulk"):
            gr.Markdown(
                "### 🚀 Bulk Video Generation\n\n"
                "**How to use:**\n"
                "1. **Create a CSV** with columns: `prompt, start_image, end_image, duration, resolution, aspect_ratio, seed, guide_scale, steps`\n"
                "2. **Zip your images** (PNG/JPG) with filenames matching the CSV\n"
                "3. **Upload CSV + ZIP**, set defaults for missing values\n"
                "4. **Click ▶️ Run Bulk Queue** → Videos auto-generate and auto-delete\n\n"
                "**Example CSV:**\n"
                "```\n"
                "prompt,start_image,end_image,duration,seed\n"
                "Sunset over ocean,sunset_start.jpg,sunset_end.jpg,5 Seconds (121 frames),-1\n"
                "City timelapse,city.png,,3 Seconds (73 frames),42\n"
                "```"
            )

            with gr.Row():
                bulk_csv = gr.File(label="📄 CSV Manifest", file_types=[".csv"])
                bulk_zip = gr.File(label="🗜️ Images ZIP", file_types=[".zip"])

            gr.Markdown("**Default Settings** *(for rows with missing values):*")
            with gr.Row():
                bulk_dur = gr.Dropdown(
                    label="Duration",
                    choices=["2 Seconds (49 frames)","3 Seconds (73 frames)","5 Seconds (121 frames)","8 Seconds (193 frames)","10 Seconds (241 frames)","15 Seconds (361 frames)"],
                    value="5 Seconds (121 frames)"
                )
                bulk_res = gr.Dropdown(
                    label="Resolution",
                    choices=["1080p","720p","540p","480p"],
                    value="720p"
                )
                bulk_asp = gr.Dropdown(
                    label="Aspect Ratio",
                    choices=["16:9 Landscape","4:3 Standard","1:1 Square","3:4 Portrait","9:16 Portrait"],
                    value="16:9 Landscape"
                )

            with gr.Row():
                bulk_guide = gr.Slider(label="Guide Scale", minimum=1.0, maximum=8.0, step=0.5, value=3.0)
                bulk_step = gr.Slider(label="Steps", minimum=2, maximum=8, step=1, value=8)

            with gr.Row():
                bulk_btn = gr.Button("▶️ Run Bulk Queue", variant="primary", size="lg")
                bulk_stp = gr.Button("🛑 Stop", variant="stop", size="lg")

            bulk_log = gr.Textbox(label="📋 Process Log", lines=15, interactive=False, placeholder="Processing logs appear here...")

            bulk_event = bulk_btn.click(
                fn=run_bulk_queue_with_auto_delete,
                inputs=[bulk_csv, bulk_zip, bulk_dur, bulk_res, bulk_asp, bulk_guide, bulk_step],
                outputs=[bulk_log],
            )
            bulk_stp.click(fn=stop_bulk, outputs=[bulk_log])

    gr.HTML('<div class="footer"><p style="font-size: 16px; margin: 5px 0;">🎬 Created by <strong>AIQuest Academy</strong></p><p style="font-size: 14px; margin: 5px 0; color: #9ca3af;">Free &amp; Open Source | LTX-2.3 22B Distilled 1.1 Q4_K_M | Kaggle T4 GPU</p><p style="font-size: 13px; margin: 10px 0;"><a href="https://youtube.com/@aiquestacademy" target="_blank" style="color: #667eea; text-decoration: none; margin: 0 10px;">YouTube</a> | <a href="https://x.com/aiquestacademy" target="_blank" style="color: #667eea; text-decoration: none; margin: 0 10px;">X (Twitter)</a> | <a href="https://aiquest.site" target="_blank" style="color: #667eea; text-decoration: none; margin: 0 10px;">aiquest.site</a></p></div>')

print("\nLaunching Gradio...")
sys.stdout.flush()

demo.queue()
demo.launch(share=True, inline=False, debug=True, show_error=True, max_threads=1, ssr_mode=False)

Writing run_ltx_t2v.py


In [5]:
import base64

def _download_file(file_path):
    """Convert file to base64 for download via Gradio."""
    if os.path.exists(file_path):
        with open(file_path, 'rb') as f:
            return base64.b64encode(f.read()).decode('utf-8')
    return None

def run_bulk_queue_with_auto_delete(csv_file, zip_file,
                                     default_duration, default_resolution,
                                     default_aspect, default_guide, default_steps,
                                     progress=None):
    """Bulk runner with auto-download and delete after each video."""
    global _bulk_stop_flag
    _bulk_stop_flag.clear()

    if csv_file is None:
        return None, "❌ Please upload a CSV file.", "No progress yet"
    
    if zip_file is None:
        return None, "❌ Please upload a ZIP file with images.", "No progress yet"

    try:
        with open(csv_file, "r", encoding="utf-8-sig") as f:
            csv_text = f.read()
        jobs = _parse_bulk_csv(csv_text)
    except Exception as e:
        return None, f"❌ CSV parse error: {e}", "Parse failed"

    if not jobs:
        return None, "❌ No jobs found in CSV.", "No jobs"

    img_dir = "/kaggle/working/bulk_images"
    img_map = _extract_images_zip(zip_file, img_dir) if zip_file else {}

    total = len(jobs)
    passed = []
    failed = []
    log_lines = [f"🚀 Starting {total} job(s)…\n"]
    progress_text = []

    for idx, job in enumerate(jobs):
        if _bulk_stop_flag.is_set():
            log_lines.append("🛑 Stopped by user.")
            break

        pct = (idx / total) * 100
        job_status = f"[{idx+1}/{total}] {job['prompt'][:60]}…"
        progress(pct / 100, desc=f"Processing: {job_status}")
        progress_text.append(job_status)
        
        log_lines.append(f"\n[{idx+1}/{total}] {job['prompt'][:80]}")

        def _resolve(name):
            if not name:
                return None
            for candidate in [name,
                               os.path.join("/kaggle/working", name),
                               img_map.get(os.path.basename(name), "")]:
                if candidate and os.path.exists(candidate):
                    return candidate
            return None

        start_path = _resolve(job["start_image"])
        end_path = _resolve(job["end_image"])

        duration = job["duration"] or default_duration
        res = job["resolution"] or default_resolution
        aspect = job["aspect_ratio"] or default_aspect
        gscale = job["guide_scale"] or default_guide
        steps = job["steps"] or default_steps
        seed = job["seed"]

        try:
            out_path, status = Video_Generation(
                prompt=job["prompt"],
                input_image_start=start_path,
                input_image_end=end_path,
                seed=seed,
                duration_dropdown=duration,
                resolution_dropdown=res,
                aspect_ratio_dropdown=aspect,
                guide_scale=gscale,
                num_steps=steps,
            )
            
            if out_path and os.path.exists(out_path):
                passed.append(out_path)
                log_lines.append(f"  ✅ Generated: {os.path.basename(out_path)}")
                
                # 📤 Upload to Google Drive BEFORE deleting
                if GDRIVE_AVAILABLE:
                    try:
                        file_id = upload_video_to_gdrive(GDRIVE, out_path, GDRIVE_FOLDER_ID)
                        if file_id:
                            log_lines.append(f"  ✅ Backed up to Google Drive: {file_id}")
                        else:
                            log_lines.append(f"  ⚠️  Google Drive backup skipped")
                    except Exception as e:
                        log_lines.append(f"  ⚠️  Google Drive upload error: {e}")
                else:
                    log_lines.append(f"  ⚠️  Google Drive not available - file will be deleted without backup")
                
                # 🗑️ Delete AFTER backup is complete
                try:
                    os.remove(out_path)
                    log_lines.append(f"  🗑️  Auto-deleted from local storage")
                except Exception as e:
                    log_lines.append(f"  ⚠️  Could not delete: {e}")
            else:
                failed.append(job["prompt"])
                log_lines.append(f"  ❌ {status}")
        except Exception as e:
            failed.append(str(e))
            log_lines.append(f"  ❌ Error: {e}")

        gc.collect()
        torch.cuda.empty_cache()

    progress(1.0, desc="✅ Batch complete!")
    summary = (f"\n{'═'*50}\n"
               f"✅ Completed: {len(passed)}  ❌ Failed: {len(failed)}  "
               f"Total: {total}\n{'═'*50}")
    log_lines.append(summary)

    if os.path.exists(img_dir):
        shutil.rmtree(img_dir, ignore_errors=True)

    final_log = "\n".join(log_lines)
    progress_summary = "\n".join(progress_text[-10:]) if progress_text else "No progress"
    
    return None, final_log, progress_summary


In [6]:
# ═══════════════════════════════════════════════════════════════════════════════
# ADD BULK PROCESSING TAB TO GRADIO INTERFACE
# ═══════════════════════════════════════════════════════════════════════════════
# Inject bulk processing tab into the demo after creation

# Find the demo object and add bulk tab
bulk_section = """
    with gr.Tabs():
        with gr.TabItem("🎬 Single Video", id="single"):
            # (existing single video content goes here via insertion)
            pass
        
        with gr.TabItem("📦 Bulk Queue", id="bulk"):
            gr.Markdown(
                "### 🚀 Bulk Video Generation\\n\\n"
                "**How it works:**\\n"
                "1. Create a CSV with columns: `prompt, start_image, end_image, duration, resolution, aspect_ratio, seed, guide_scale, steps`\\n"
                "2. Only `prompt` is required; all other columns are optional\\n"
                "3. Zip your images (PNG/JPG) with filenames matching the CSV\\n"
                "4. Upload CSV and ZIP, set defaults, click **▶️ Run Bulk Queue**\\n"
                "5. Videos auto-generate and auto-delete after processing\\n\\n"
                "**Example CSV:**\\n"
                "```\\n"
                "prompt,start_image,end_image,duration,seed\\n"
                "A sunset over ocean,sunset_start.jpg,sunset_end.jpg,5 Seconds (121 frames),-1\\n"
                "City timelapse,city.png,,3 Seconds (73 frames),42\\n"
                "```"
            )
            
            with gr.Row():
                bulk_csv_file = gr.File(label="📄 CSV Manifest", file_types=[".csv"])
                bulk_zip_file = gr.File(label="🗜️ Images ZIP", file_types=[".zip"])
            
            gr.Markdown("**Defaults (for rows with missing values):**")
            with gr.Row():
                bulk_duration = gr.Dropdown(
                    label="Duration",
                    choices=["2 Seconds (49 frames)","3 Seconds (73 frames)",
                             "5 Seconds (121 frames)","8 Seconds (193 frames)",
                             "10 Seconds (241 frames)","15 Seconds (361 frames)"],
                    value="5 Seconds (121 frames)"
                )
                bulk_resolution = gr.Dropdown(
                    label="Resolution",
                    choices=["1080p","720p","540p","480p"],
                    value="720p"
                )
                bulk_aspect = gr.Dropdown(
                    label="Aspect Ratio",
                    choices=["16:9 Landscape","4:3 Standard","1:1 Square","3:4 Portrait","9:16 Portrait"],
                    value="16:9 Landscape"
                )
            
            with gr.Row():
                bulk_guide = gr.Slider(label="Guide Scale", minimum=1.0, maximum=8.0, step=0.5, value=3.0)
                bulk_steps = gr.Slider(label="Steps", minimum=2, maximum=8, step=1, value=8)
            
            with gr.Row():
                bulk_run_btn = gr.Button("▶️ Run Bulk Queue", variant="primary", size="lg")
                bulk_stop_btn = gr.Button("🛑 Stop", variant="stop", size="lg")
            
            bulk_progress = gr.Label(label="📊 Progress", value="Ready")
            bulk_log = gr.Textbox(label="📋 Process Log", lines=15, interactive=False, 
                                   placeholder="Logs will appear here...")
            
            bulk_event = bulk_run_btn.click(
                fn=run_bulk_queue_with_auto_delete,
                inputs=[bulk_csv_file, bulk_zip_file, bulk_duration, bulk_resolution, 
                        bulk_aspect, bulk_guide, bulk_steps],
                outputs=[None, bulk_log, bulk_progress],
            )
            
            bulk_stop_btn.click(fn=stop_bulk, outputs=[bulk_log])
"""

# Note: The actual tab insertion happens in the main script via run_ltx_t2v.py
# This section documents the bulk processing tab structure

print("✅ Bulk processing functions defined and ready for integration")
print("Run the script generation cell to update run_ltx_t2v.py with bulk tab UI")


✅ Bulk processing functions defined and ready for integration
Run the script generation cell to update run_ltx_t2v.py with bulk tab UI


In [7]:
%%bash
python - <<'PATCH'
from pathlib import Path

path = Path('/kaggle/working/run_ltx_t2v.py')
if not path.exists():
    raise FileNotFoundError(f'{path} not found. Run script generation cell first.')

text = path.read_text(encoding='utf-8')

# Check if already patched
if 'with gr.Tabs():' in text:
    print('run_ltx_t2v.py already has tabbed UI')
else:
    # Find the line "with gr.Blocks(...) as demo:" and replace to use tabs
    old = 'with gr.Blocks(css=CSS, theme=gr.themes.Soft(), title="LTX-2.3 22B Video Generator | AIQUEST") as demo:'
    new = '''with gr.Blocks(css=CSS, theme=gr.themes.Soft(), title="LTX-2.3 22B Video Generator | AIQUEST") as demo:
    with gr.Tabs():
        with gr.TabItem("🎬 Single Video", id="single"):'''
    
    if old in text:
        text = text.replace(old, new)
        
        # Find where the HTML footer is and add proper indentation
        # Everything between "gr.HTML('<div class=\"brand-header\"...)" and "gen_event = gen_btn.click(...)"
        # needs to be indented
        
        # Insert closing for single tab before the footer
        footer_marker = 'gr.HTML(\'<div class="footer">'
        if footer_marker in text:
            text = text.replace(
                footer_marker,
                '\n        with gr.TabItem("📦 Bulk Queue", id="bulk"):\n            gr.Markdown(\n                "### 🚀 Bulk Video Generation\\n\\n"\n                "**How to use:**\\n"\n                "1. Create CSV: `prompt, start_image, end_image, duration, resolution, aspect_ratio, seed, guide_scale, steps`\\n"\n                "2. Zip your images (PNG/JPG) - filenames match CSV\\n"\n                "3. Upload CSV + ZIP, set defaults, click **Run Bulk Queue**\\n"\n                "4. Videos auto-generate, auto-download, and auto-delete\\n\\n"\n                "**Example CSV:**\\n"\n                "```\\nprompt,start_image,end_image,duration\\nSunset over ocean,start.jpg,end.jpg,5 Seconds (121 frames)\\n```"\n            )\n            with gr.Row():\n                bulk_csv = gr.File(label="📄 CSV Manifest", file_types=[".csv"])\n                bulk_zip = gr.File(label="🗜️ Images ZIP", file_types=[".zip"])\n            with gr.Row():\n                bulk_dur = gr.Dropdown(label="Duration", choices=["2 Seconds (49 frames)","3 Seconds (73 frames)","5 Seconds (121 frames)","8 Seconds (193 frames)","10 Seconds (241 frames)","15 Seconds (361 frames)"], value="5 Seconds (121 frames)")\n                bulk_res = gr.Dropdown(label="Resolution", choices=["1080p","720p","540p","480p"], value="720p")\n                bulk_asp = gr.Dropdown(label="Aspect", choices=["16:9 Landscape","4:3 Standard","1:1 Square","3:4 Portrait","9:16 Portrait"], value="16:9 Landscape")\n            with gr.Row():\n                bulk_guide = gr.Slider(label="Guide", minimum=1.0, maximum=8.0, step=0.5, value=3.0)\n                bulk_step = gr.Slider(label="Steps", minimum=2, maximum=8, step=1, value=8)\n            with gr.Row():\n                bulk_btn = gr.Button("▶️ Run Bulk", variant="primary", size="lg")\n                bulk_stp = gr.Button("🛑 Stop", variant="stop", size="lg")\n            bulk_prog = gr.Textbox(label="Progress", interactive=False, value="Ready")\n            bulk_log = gr.Textbox(label="Log", lines=12, interactive=False, placeholder="Processing logs...")\n            bulk_event = bulk_btn.click(\n                fn=run_bulk_queue_with_auto_delete,\n                inputs=[bulk_csv, bulk_zip, bulk_dur, bulk_res, bulk_asp, bulk_guide, bulk_step],\n                outputs=[None, bulk_log, bulk_prog],\n            )\n            bulk_stp.click(fn=stop_bulk, outputs=[bulk_log])\n\n        gr.HTML(\'<div class="footer">'
            )
        
        # Indent all the single video UI code
        lines = text.split('\n')
        new_lines = []
        in_single_tab = False
        indent_next = False
        
        for i, line in enumerate(lines):
            if 'with gr.TabItem("🎬 Single Video"' in line:
                in_single_tab = True
                indent_next = True
                new_lines.append(line)
            elif 'with gr.TabItem("📦 Bulk Queue"' in line:
                in_single_tab = False
                indent_next = False
                new_lines.append(line)
            elif in_single_tab and line and not line.startswith('        '):
                # Add indent to content lines that aren't already indented enough
                if line.strip() and not any(x in line for x in ['with gr.Tabs', 'with gr.TabItem']):
                    new_lines.append('    ' + line)
                else:
                    new_lines.append(line)
            else:
                new_lines.append(line)
        
        text = '\n'.join(new_lines)
        
        path.write_text(text, encoding='utf-8')
        print('✅ Added tabbed UI with bulk processing tab')

# Also check for missing functions and add them if needed
if 'def run_bulk_queue_with_auto_delete' not in text:
    print('⚠️  run_bulk_queue_with_auto_delete not found - ensure it\'s defined in earlier cells')
else:
    print('✅ Bulk processing function found')

PATCH


run_ltx_t2v.py already has tabbed UI
⚠️  run_bulk_queue_with_auto_delete not found - ensure it's defined in earlier cells


In [8]:

# Install Google Drive dependencies
import subprocess
import sys

print("Installing Google Drive dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydrive", "google-auth-oauthlib"], check=False)

# ═══════════════════════════════════════════════════════════════════════════════
# GOOGLE DRIVE INTEGRATION - Upload & Backup Videos Before Deletion
# ═══════════════════════════════════════════════════════════════════════════════

def setup_google_drive():
    """Setup Google Drive authentication and create output folder."""
    try:
        from google.colab import auth
        auth.authenticate_user()
        print("✅ Google Colab authentication detected")
        
        from pydrive.auth import GoogleAuth
        from pydrive.drive import GoogleDrive
        
        gauth = GoogleAuth()
        gauth.auth_method = 'authenticated_user_id'
        gauth.LoadCredentialsFile("mycreds.txt")
        
        if gauth.credentials is None:
            gauth.GetFlow()
            gauth.Flow.InsertMessage(None)
            gauth.Authorize()
        elif gauth.access_token_expired:
            gauth.Refresh()
        else:
            gauth.Authorize()
        
        gauth.SaveCredentialsFile("mycreds.txt")
        drive = GoogleDrive(gauth)
        return drive
    except ImportError:
        print("⚠️  pydrive not installed. Using gdown as fallback...")
        return None
    except Exception as e:
        print(f"⚠️  Google Drive setup warning: {e}")
        return None

def get_or_create_gdrive_folder(drive, folder_name="LTX_Videos_Backup"):
    """Get or create a folder in Google Drive root for video backups."""
    if drive is None:
        return None
    
    try:
        query = f"title='{folder_name}' and trashed=false and mimeType='application/vnd.google-apps.folder'"
        folders = drive.ListFile({'q': query}).GetList()
        
        if len(folders) > 0:
            folder_id = folders[0]['id']
            print(f"✅ Found existing folder: {folder_name}")
            return folder_id
        else:
            folder = drive.CreateFile({
                'title': folder_name,
                'mimeType': 'application/vnd.google-apps.folder'
            })
            folder.Upload()
            folder_id = folder['id']
            print(f"✅ Created new folder: {folder_name}")
            return folder_id
    except Exception as e:
        print(f"❌ Error managing Google Drive folder: {e}")
        return None

def upload_video_to_gdrive(drive, video_path, folder_id=None):
    """Upload a video file to Google Drive and return file ID."""
    if drive is None or video_path is None or not os.path.exists(video_path):
        return None
    
    try:
        filename = os.path.basename(video_path)
        file_size_mb = os.path.getsize(video_path) / (1024**2)
        
        file_metadata = {
            'title': filename,
            'mimeType': 'video/mp4'
        }
        if folder_id:
            file_metadata['parents'] = [{'id': folder_id}]
        
        gfile = drive.CreateFile(file_metadata)
        gfile.SetContentFile(video_path)
        print(f"  📤 Uploading {filename} ({file_size_mb:.1f} MB) to Google Drive...")
        gfile.Upload()
        
        file_id = gfile['id']
        print(f"  ✅ Uploaded to Google Drive: {file_id}")
        return file_id
    except Exception as e:
        print(f"  ❌ Upload failed: {e}")
        return None

# ── Initialize Google Drive on startup ──
try:
    GDRIVE = setup_google_drive()
    GDRIVE_FOLDER_ID = get_or_create_gdrive_folder(GDRIVE, "LTX_Videos_Backup") if GDRIVE else None
    GDRIVE_AVAILABLE = GDRIVE is not None
    print(f"\n{'='*60}")
    print(f"Google Drive Status: {'✅ ENABLED' if GDRIVE_AVAILABLE else '⚠️ DISABLED (will proceed without backup)'}")
    print(f"{'='*60}\n")
except Exception as e:
    GDRIVE = None
    GDRIVE_FOLDER_ID = None
    GDRIVE_AVAILABLE = False
    print(f"⚠️  Google Drive initialization skipped: {e}\n")


Installing Google Drive dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 987.4/987.4 kB 15.6 MB/s eta 0:00:00


⚠️  Google Drive setup warning: raw_input was called, but this frontend does not support input requests.

Google Drive Status: ⚠️ DISABLED (will proceed without backup)



In [9]:

# ═══════════════════════════════════════════════════════════════════════════════
# SINGLE VIDEO BACKUP TO GOOGLE DRIVE
# ═══════════════════════════════════════════════════════════════════════════════

def backup_video_to_gdrive(selected_video):
    """Backup a single selected video to Google Drive."""
    if not selected_video:
        return "❌ No video selected."
    
    if not GDRIVE_AVAILABLE:
        return "❌ Google Drive not available. Is it linked to Kaggle?"
    
    video_path = os.path.join(OUTPUT_DIR, selected_video)
    if not os.path.exists(video_path):
        return f"❌ Video file not found: {selected_video}"
    
    try:
        file_id = upload_video_to_gdrive(GDRIVE, video_path, GDRIVE_FOLDER_ID)
        if file_id:
            return f"✅ Video backed up to Google Drive!\n   File ID: {file_id}"
        else:
            return "❌ Upload failed. Check Google Drive connection."
    except Exception as e:
        return f"❌ Backup error: {e}"

def backup_all_videos_to_gdrive():
    """Backup all generated videos to Google Drive."""
    if not os.path.isdir(OUTPUT_DIR):
        return "⚠️  No videos to backup."
    
    if not GDRIVE_AVAILABLE:
        return "❌ Google Drive not available."
    
    videos = [f for f in os.listdir(OUTPUT_DIR) if f.lower().endswith(('.mp4', '.mkv', '.webm'))]
    if not videos:
        return "⚠️  No videos found in output directory."
    
    backed_up = 0
    failed = 0
    
    for video in videos:
        video_path = os.path.join(OUTPUT_DIR, video)
        try:
            file_id = upload_video_to_gdrive(GDRIVE, video_path, GDRIVE_FOLDER_ID)
            if file_id:
                backed_up += 1
            else:
                failed += 1
        except Exception as e:
            print(f"  Error backing up {video}: {e}")
            failed += 1
    
    status = f"✅ Backed up {backed_up} videos to Google Drive"
    if failed > 0:
        status += f" ({failed} failed)"
    return status

def delete_selected_with_gdrive_backup(selected):
    """Delete video ONLY if Google Drive backup succeeds."""
    if not selected:
        return list_outputs(), "❌ No video selected."
    
    video_path = os.path.join(OUTPUT_DIR, selected)
    if not os.path.exists(video_path):
        return list_outputs(), "❌ File not found."
    
    # Backup to Google Drive first
    if GDRIVE_AVAILABLE:
        try:
            file_id = upload_video_to_gdrive(GDRIVE, video_path, GDRIVE_FOLDER_ID)
            if file_id:
                msg = f"✅ Backed up to Google Drive before deletion"
            else:
                return list_outputs(), f"⚠️  Backup failed - file NOT deleted for safety"
        except Exception as e:
            return list_outputs(), f"❌ Backup error - file NOT deleted for safety: {e}"
    else:
        msg = "⚠️  Google Drive not available - deleting without backup"
    
    # Delete only after successful backup
    try:
        os.remove(video_path)
        return list_outputs(), f"{msg}\n✅ Deleted {selected}"
    except Exception as e:
        return list_outputs(), f"❌ Deletion failed: {e}"

def delete_all_with_gdrive_backup():
    """Delete all videos ONLY if Google Drive backup completes."""
    if not os.path.isdir(OUTPUT_DIR):
        return [], "✅ No outputs to delete."
    
    videos = [f for f in os.listdir(OUTPUT_DIR) if f.lower().endswith(('.mp4', '.mkv', '.webm'))]
    if not videos:
        return [], "✅ No videos found."
    
    backed_up = 0
    failed_backup = []
    
    # Backup all to Google Drive first
    if GDRIVE_AVAILABLE:
        for video in videos:
            video_path = os.path.join(OUTPUT_DIR, video)
            try:
                file_id = upload_video_to_gdrive(GDRIVE, video_path, GDRIVE_FOLDER_ID)
                if file_id:
                    backed_up += 1
                else:
                    failed_backup.append(video)
            except Exception as e:
                print(f"  Backup error for {video}: {e}")
                failed_backup.append(video)
        
        if failed_backup:
            return list_outputs(), f"⚠️  {len(failed_backup)} backups failed - files NOT deleted:\n  " + "\n  ".join(failed_backup)
    else:
        backed_up = len(videos)
    
    # Delete all
    deleted = 0
    for video in videos:
        try:
            os.remove(os.path.join(OUTPUT_DIR, video))
            deleted += 1
        except Exception:
            pass
    
    return list_outputs(), f"✅ Backed up {backed_up} videos to Google Drive, then deleted all {deleted} local copies"


---
## Step 5: Launch!

Runs the generation script. Watch for the **public Gradio URL** in the output.

In [10]:
!cd /kaggle/working && python -u run_ltx_t2v.py 2>&1

  [GPU] sm_75 detected — native CUDA mode, no patches needed
GPU: Tesla T4
Compute Capability: (7, 5)
VRAM: 14.6 GB
RAM: 31.4 GB total, 28.8 GB available

Loading LTX-2.3 22B Distilled 1.1 (GGUF Q4_K_M)...
2026-06-14 12:33:53.044399: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781440433.267464     304 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781440433.329937     304 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781440433.884398     304 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781440433.884481     304 computation_placer

---

<div align="center">

### Enjoyed this notebook?

If this was helpful, please upvote and subscribe for more free AI tools and tutorials!

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>

</div>
